<a href="https://colab.research.google.com/github/predicting-pregnancy-success/pregnancy-ml-models/blob/main/%EB%82%9C%EC%9E%84_%ED%99%98%EC%9E%90_%EB%8C%80%EC%83%81_%EC%9E%84%EC%8B%A0_%EC%84%B1%EA%B3%B5_%EC%97%AC%EB%B6%80_%EC%98%88%EC%B8%A1AI_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd
import numpy as np
import random
import os

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [37]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

In [38]:
# 1. 테스트 로드

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

display(train.head())

,ID,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
0,TRAIN_000000,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0
1,TRAIN_000001,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
2,TRAIN_000002,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,2.0,NaN,0
3,TRAIN_000003,TRJXFG,만35-37세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
4,TRAIN_000004,TRVNRY,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0


In [4]:
# 전체 데이터 확인

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256351 entries, 0 to 256350
Data columns (total 69 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     256351 non-null  object 
 1   시술 시기 코드               256351 non-null  object 
 2   시술 당시 나이               256351 non-null  object 
 3   임신 시도 또는 마지막 임신 경과 연수  9370 non-null    float64
 4   시술 유형                  256351 non-null  object 
 5   특정 시술 유형               256349 non-null  object 
 6   배란 자극 여부               256351 non-null  int64  
 7   배란 유도 유형               256351 non-null  object 
 8   단일 배아 이식 여부            250060 non-null  float64
 9   착상 전 유전 검사 사용 여부       2718 non-null    float64
 10  착상 전 유전 진단 사용 여부       250060 non-null  float64
 11  남성 주 불임 원인             256351 non-null  int64  
 12  남성 부 불임 원인             256351 non-null  int64  
 13  여성 주 불임 원인             256351 non-null  int64  
 14  여성 부 불임 원인             256351 non-nu

In [5]:
train.isnull().sum()

,0
ID,0
시술 시기 코드,0
시술 당시 나이,0
임신 시도 또는 마지막 임신 경과 연수,246981
시술 유형,0
...,...
난자 해동 경과일,254915
난자 혼합 경과일,53735
배아 이식 경과일,43566
배아 해동 경과일,215982


In [6]:
train.describe()

,임신 시도 또는 마지막 임신 경과 연수,배란 자극 여부,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,착상 전 유전 진단 사용 여부,남성 주 불임 원인,남성 부 불임 원인,여성 주 불임 원인,여성 부 불임 원인,부부 주 불임 원인,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
count,9370.000000,256351.000000,250060.000000,2718.0,250060.000000,256351.000000,256351.000000,256351.000000,256351.000000,256351.000000,...,250060.000000,250060.000000,2179.0,1929.0,198863.0,1436.000000,202616.000000,212785.000000,40369.000000,256351.000000
mean,9.270651,0.771286,0.233476,1.0,0.012781,0.028516,0.013115,0.030724,0.012432,0.033068,...,0.009830,0.004195,1.0,1.0,0.0,0.001393,0.005385,3.254741,0.045629,0.258349
std,3.550313,0.420005,0.423043,0.0,0.112328,0.166441,0.113767,0.172568,0.110805,0.178814,...,0.098656,0.064633,0.0,0.0,0.0,0.037307,0.111504,1.715697,0.418672,0.437728
min,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.000000,1.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,1.0,0.0,0.000000,0.000000,2.000000,0.000000,0.000000
50%,9.000000,1.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,1.0,0.0,0.000000,0.000000,3.000000,0.000000,0.000000
75%,11.000000,1.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.0,1.0,0.0,0.000000,0.000000,5.000000,0.000000,1.000000
max,20.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.0,1.0,0.0,1.000000,7.000000,7.000000,7.000000,1.000000


In [7]:
# 결측치 처리 (v3 — 결측 indicator 강화)
train_copy = train.copy()
test_copy  = test.copy()

# '임신 성공 여부' 결측치 행만 찾아서 삭제하기
train_copy = train_copy.dropna(subset=['임신 성공 여부']).reset_index(drop=True)

# 결측치가 너무 많은 컬럼 삭제하기
train_copy = train_copy.drop(columns=['임신 시도 또는 마지막 임신 경과 연수'])
test_copy  = test_copy.drop(columns=['임신 시도 또는 마지막 임신 경과 연수'])

# === [변경 1] 모든 NaN 보유 컬럼에 _missing indicator 추가 ===
# (NaN을 채우기 전에 먼저 만들어야 정보 보존됨)
nan_cols = [
    c for c in train_copy.columns
    if c not in ['ID', '임신 성공 여부']
    and (train_copy[c].isna().any() or (c in test_copy.columns and test_copy[c].isna().any()))
]
print(f"NaN 보유 컬럼: {len(nan_cols)}개")

for col in nan_cols:
    train_copy[f'{col}_missing'] = train_copy[col].isna().astype(np.int8)
    test_copy[f'{col}_missing']  = test_copy[col].isna().astype(np.int8)

# === [변경 2] 교차 결측 시그니처 ===
# 전체 NaN 개수 → 시술 유형 시그니처
train_copy['nan_count'] = train_copy[nan_cols].isna().sum(axis=1).astype(np.int16)
test_copy['nan_count']  = test_copy[nan_cols].isna().sum(axis=1).astype(np.int16)

# IVF 미세주입 NaN 개수 → DI vs IVF 구분
ivf_micro = ['미세주입된 난자 수', '미세주입에서 생성된 배아 수',
             '미세주입 후 저장된 배아 수', '미세주입 배아 이식 수']
ivf_micro = [c for c in ivf_micro if c in nan_cols]
if ivf_micro:
    train_copy['ivf_nan_count'] = train_copy[ivf_micro].isna().sum(axis=1).astype(np.int8)
    test_copy['ivf_nan_count']  = test_copy[ivf_micro].isna().sum(axis=1).astype(np.int8)

# 배아/난자 처리 NaN 시그니처
embryo_proc = ['총 생성 배아 수', '이식된 배아 수', '저장된 배아 수',
               '해동된 배아 수', '수집된 신선 난자 수', '혼합된 난자 수']
embryo_proc = [c for c in embryo_proc if c in nan_cols]
if embryo_proc:
    train_copy['embryo_nan_count'] = train_copy[embryo_proc].isna().sum(axis=1).astype(np.int8)
    test_copy['embryo_nan_count']  = test_copy[embryo_proc].isna().sum(axis=1).astype(np.int8)

# 유전 검사 NaN 시그니처
genetic = ['착상 전 유전 검사 사용 여부', '착상 전 유전 진단 사용 여부',
           'PGD 시술 여부', 'PGS 시술 여부']
genetic = [c for c in genetic if c in nan_cols]
if genetic:
    train_copy['genetic_nan_count'] = train_copy[genetic].isna().sum(axis=1).astype(np.int8)
    test_copy['genetic_nan_count']  = test_copy[genetic].isna().sum(axis=1).astype(np.int8)

# === [기존] 결측 매우 많은 컬럼 → 0 ===
very_high_missing = [
    '착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부',
    '난자 해동 경과일', '난자 채취 경과일', '난자 혼합 경과일',
    '배아 이식 경과일', '배아 해동 경과일',
]
for col in very_high_missing:
    train_copy[col] = train_copy[col].fillna(0)
    test_copy[col]  = test_copy[col].fillna(0)

# === [기존] Unknown으로 채울 카테고리 컬럼 ===
object_cols = [
    '특정 시술 유형', '배아 생성 주요 이유', '총 시술 횟수', '클리닉 내 총 시술 횟수',
    'IVF 시술 횟수', 'DI 시술 횟수', '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수', '난자 출처', '정자 출처',
    '난자 기증자 나이', '정자 기증자 나이'
]
train_copy[object_cols] = train_copy[object_cols].fillna('Unknown')
test_copy[object_cols]  = test_copy[object_cols].fillna('Unknown')

# === [기존] 0으로 채울 숫자형 컬럼 ===
number_cols = [
    '단일 배아 이식 여부', '착상 전 유전 진단 사용 여부',
    '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '총 생성 배아 수',
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수',
    '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수',
    '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수',
    '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
    '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부'
]
train_copy[number_cols] = train_copy[number_cols].fillna(0)
test_copy[number_cols]  = test_copy[number_cols].fillna(0)

print(f" 신규 _missing 컬럼: {len(nan_cols)}개")
print(f" 신규 시그니처: nan_count, ivf_nan_count, embryo_nan_count, genetic_nan_count")
print(f" train_copy shape: {train_copy.shape}")
print(f" test_copy shape:  {test_copy.shape}")

NaN 보유 컬럼: 30개
 신규 _missing 컬럼: 30개
 신규 시그니처: nan_count, ivf_nan_count, embryo_nan_count, genetic_nan_count
 train_copy shape: (256351, 102)
 test_copy shape:  (90067, 101)


In [8]:
# 결측치 처리 후 확인
print(train_copy.isnull().sum().sum())
print(test_copy.isnull().sum().sum())

train_copy.info()

0
0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256351 entries, 0 to 256350
Columns: 102 entries, ID to genetic_nan_count
dtypes: float64(28), int16(1), int64(19), int8(33), object(21)
memory usage: 141.6+ MB


In [9]:
# 인코딩

# 시술 당시 나이 구간
age_map = {
    '만18-34세': 0, '만35-37세': 1, '만38-39세': 2,
    '만40-42세': 3, '만43-44세': 4, '만45-50세': 5,
    '알 수 없음': -1, 'Unknown': -1
}

# 시술 당시 나이 맵핑
train_copy['시술 당시 나이'] = train_copy['시술 당시 나이'].map(age_map)
test_copy['시술 당시 나이'] = test_copy['시술 당시 나이'].map(age_map)


# 기증자 나이 구간
donor_map = {
    '만20세 이하': 0, '만21-25세': 1, '만26-30세': 2,
    '만31-35세': 3, '만36-40세': 4, '만41-45세': 5,
    '알 수 없음': -1, 'Unknown': -1
}

# 기증자 나이 맵핑 (난자, 정자 둘 다)
for col in ['난자 기증자 나이', '정자 기증자 나이']:
    train_copy[col] = train_copy[col].map(donor_map)
    test_copy[col] = test_copy[col].map(donor_map)


# 횟수 구간 (0회 ~ 6회 이상)
count_map = {
    '0회': 0, '1회': 1, '2회': 2, '3회': 3,
    '4회': 4, '5회': 5, '6회 이상': 6,
    'Unknown': -1
}

# 횟수와 관련된 컬럼들 리스트
count_cols = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
]

# 횟수 관련 모든 컬럼 맵핑
for col in count_cols:
    train_copy[col] = train_copy[col].map(count_map)
    test_copy[col] = test_copy[col].map(count_map)


In [10]:
# 명목형 인코딩
nominal_cols = [
    '시술 시기 코드', '시술 유형', '특정 시술 유형',
    '배란 유도 유형', '배아 생성 주요 이유',
    '난자 출처', '정자 출처',
]

for col in nominal_cols:
    le = LabelEncoder()

    # 문자열로 통일
    train_copy[col] = train_copy[col].astype(str)
    test_copy[col] = test_copy[col].astype(str)

    # train에 있는 글자 종류를 파악
    train_labels = set(train_copy[col].unique())

    # test에만 있는 글자는 'Unknown'으로 대체
    test_copy[col] = test_copy[col].apply(lambda x: x if x in train_labels else 'Unknown')

    le.fit(list(train_labels) + ['Unknown'])

    # 숫자로 변환
    train_copy[col] = le.transform(train_copy[col])
    test_copy[col] = le.transform(test_copy[col])

In [11]:
train_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256351 entries, 0 to 256350
Columns: 102 entries, ID to genetic_nan_count
dtypes: float64(28), int16(1), int64(39), int8(33), object(1)
memory usage: 141.6+ MB


<파생변수 생성 인사이트>

나이 젊을수록 성공률↑

출산경험 있으면 성공률↑

신선난자↑→ 배아수↑→ 성공률↑

혼합난자↑→ 배아수↑→ 성공률↑

이식 경과일 길수록 성공률↑

시술 횟수↑ → 성공률↓

정자/난자 본인 여부에 따른 성공 확률

In [12]:
# 파생변수 생성(260424)

def create_all_features(df):
    """
    파생변수 최종
    - 시각화 인사이트 기반 (나이/출산/효율/이식/이력)
    - 난자/정자 출처
    - 유용한 비율/효율 피처
    """
    df = df.copy()

    # 나이 구간
    df['고령_38이상']    = (df['시술 당시 나이'] >= 2).astype(int)
    df['고령_40이상']    = (df['시술 당시 나이'] >= 3).astype(int)
    df['초고령_43이상']  = (df['시술 당시 나이'] >= 4).astype(int)

    # 출산/임신 경험
    df['경산부']         = (df['총 출산 횟수'] > 0).astype(int)
    df['임신경험']       = (df['총 임신 횟수'] > 0).astype(int)
    df['유산경험_수']    = (df['총 임신 횟수'] - df['총 출산 횟수']).clip(lower=0)
    df['과거_시술성공률'] = df['총 출산 횟수'] / (df['총 시술 횟수'] + 1)

    # 난자 → 배아 효율
    df['신선난자_배아효율'] = df['총 생성 배아 수'] / (df['수집된 신선 난자 수'] + 1)
    df['신선난자_사용']    = (df['수집된 신선 난자 수'] > 0).astype(int)
    df['혼합난자_배아효율'] = df['총 생성 배아 수'] / (df['혼합된 난자 수'] + 1)
    df['난자_혼합률']     = df['혼합된 난자 수'] / (df['수집된 신선 난자 수'] + 1)
    df['파트너정자_비율']  = df['파트너 정자와 혼합된 난자 수'] / (df['혼합된 난자 수'] + 1)

    # 이식 관련
    df['이식_분할기']  = (df['배아 이식 경과일'].between(2, 3)).astype(int)
    df['이식_포배기']  = (df['배아 이식 경과일'] >= 5).astype(int)
    df['배양_기간']    = (df['배아 이식 경과일'] - df['난자 채취 경과일']).clip(lower=0)

    # 시술 이력
    df['반복실패_여부'] = ((df['총 시술 횟수'] >= 3) &
                        (df['총 출산 횟수'] == 0)).astype(int)
    df['실패경험_수']   = (df['총 시술 횟수'] - df['총 출산 횟수']).clip(lower=0)

    # 난자/정자 출처
    df['기증난자_사용'] = (df['난자 출처'] == 1).astype(int)
    df['기증정자_사용'] = (df['정자 출처'] == 1).astype(int)
    df['실효_난자_나이'] = np.where(
        (df['난자 출처'] == 1) & (df['난자 기증자 나이'] != -1),
        df['난자 기증자 나이'],
        df['시술 당시 나이']
    )
    df['고령_기증난자_상쇄'] = ((df['시술 당시 나이'] >= 3) &
                             (df['난자 출처'] == 1)).astype(int)

    # 배아/난자 파이프라인 효율
    df['이식_비율']    = df['이식된 배아 수'] / (df['총 생성 배아 수'] + 1)
    df['배아_여유']    = df['저장된 배아 수'] / (df['총 생성 배아 수'] + 1)
    df['파이프_효율']  = df['이식된 배아 수'] / (df['수집된 신선 난자 수'] + 1)
    df['배아_손실수']  = (df['총 생성 배아 수'] - df['이식된 배아 수']).clip(lower=0)


    # 시술 방식/진행 시간 간격
    df['혼합_이식_간격']    = (df['배아 이식 경과일'] - df['난자 혼합 경과일']).abs()
    df['해동_이식_간격']    = (df['배아 이식 경과일'] - df['배아 해동 경과일']).abs()
    df['난자해동_혼합_간격'] = (df['난자 혼합 경과일'] - df['난자 해동 경과일']).abs()


    return df



# 적용


# 이전에 쓸모없는 피처가 이미 들어갔으면 먼저 제거
useless_cols = [
    '배아수_x_실효나이', '배아수_x_기증난자', '이식_배아_0', '이식_배아_2이상',
    '이식_0개', '이식_1개', '이식_2개', '이식_3개이상',
    '배아x나이', '배아x기증', '시술x나이', '시술x배아','신선_사이클',
    '동결난자_사이클', '동결배아_이식', '채취_혼합_간격', '사이클_복합도'
]
for c in useless_cols:
    if c in train_copy.columns:
        train_copy = train_copy.drop(columns=c)
    if c in test_copy.columns:
        test_copy = test_copy.drop(columns=c)

# 혹시 이미 create_all_features가 적용된 상태라면, 원본에서 다시 시작
# (이미 만든 피처를 또 만들면 에러는 없지만 안전하게 재생성)
train_copy = create_all_features(train_copy)
test_copy  = create_all_features(test_copy)

print(f"✅ 최종 Train shape: {train_copy.shape}")
print(f"✅ 최종 Test shape:  {test_copy.shape}")
print(f"\n신규 피처 목록:")
new_features = ['고령_38이상', '고령_40이상', '초고령_43이상',
                '경산부', '임신경험', '유산경험_수', '과거_시술성공률',
                '신선난자_배아효율', '신선난자_사용', '혼합난자_배아효율',
                '난자_혼합률', '파트너정자_비율',
                '이식_분할기', '이식_포배기', '배양_기간',
                '반복실패_여부', '실패경험_수',
                '기증난자_사용', '기증정자_사용', '실효_난자_나이', '고령_기증난자_상쇄',
                '이식_비율', '배아_여유', '파이프_효율', '배아_손실수',
                '혼합_이식_간격','해동_이식_간격','난자해동_혼합_간격']
print(f"총 {len(new_features)}개")
for f in new_features:
    print(f"  - {f}")

✅ 최종 Train shape: (256351, 130)
✅ 최종 Test shape:  (90067, 129)

신규 피처 목록:
총 28개
  - 고령_38이상
  - 고령_40이상
  - 초고령_43이상
  - 경산부
  - 임신경험
  - 유산경험_수
  - 과거_시술성공률
  - 신선난자_배아효율
  - 신선난자_사용
  - 혼합난자_배아효율
  - 난자_혼합률
  - 파트너정자_비율
  - 이식_분할기
  - 이식_포배기
  - 배양_기간
  - 반복실패_여부
  - 실패경험_수
  - 기증난자_사용
  - 기증정자_사용
  - 실효_난자_나이
  - 고령_기증난자_상쇄
  - 이식_비율
  - 배아_여유
  - 파이프_효율
  - 배아_손실수
  - 혼합_이식_간격
  - 해동_이식_간격
  - 난자해동_혼합_간격


In [13]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [14]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 2.5 MB/s eta 0:00:00


In [15]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# Optuna에서 찾은 최적 파라미터
best_params = {
    'learning_rate': 0.010402169777212351,
    'depth': 8,
    'l2_leaf_reg': 7.35932864462546,
    'min_data_in_leaf': 18,
    'bagging_temperature': 0.6527836420072781,
    'random_strength': 1.7526579808481118,
    'border_count': 121,
    'iterations': 3000,              # learning_rate 낮아서 더 많이 필요
    'eval_metric': 'AUC',
    'early_stopping_rounds': 150,    # 여유 있게
    'verbose': False
}

X = train_copy.drop(columns=['ID', '임신 성공 여부'])
y = train_copy['임신 성공 여부']
X_test = test_copy.drop(columns=['ID'])

# 3-seed × 5-fold
seeds = [42, 99, 2024]
tuned_test = np.zeros(len(X_test))
tuned_oof = np.zeros(len(X))

for seed in seeds:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        params = best_params.copy()
        params['random_seed'] = seed

        m = CatBoostClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X.iloc[val_idx], y.iloc[val_idx]))

        val_prob = m.predict_proba(X.iloc[val_idx])[:, 1]
        tuned_oof[val_idx] += val_prob / len(seeds)
        tuned_test += m.predict_proba(X_test)[:, 1] / (5 * len(seeds))

        auc = roc_auc_score(y.iloc[val_idx], val_prob)
        seed_aucs.append(auc)

    print(f"Seed {seed} 평균: {np.mean(seed_aucs):.4f}")

print(f"\n▶ 튜닝 최종 OOF AUC: {roc_auc_score(y, tuned_oof):.4f}")
print(f"▶ 이전 최고 OOF:      0.7403")

Seed 42 평균: 0.7402
Seed 99 평균: 0.7401
Seed 2024 평균: 0.7403

▶ 튜닝 최종 OOF AUC: 0.7404
▶ 이전 최고 OOF:      0.7403


In [16]:
!pip install lightgbm -q

In [17]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# CatBoost와 동일한 X, y 사용 (이미 train_copy/test_copy에서 만들어진 상태)
# 만약 변수가 사라졌으면 아래 주석 해제
# X = train_copy.drop(columns=['ID', '임신 성공 여부'])
# y = train_copy['임신 성공 여부'].astype(int)
# X_test = test_copy.drop(columns=['ID'])

# LightGBM 파라미터 (난임 데이터에 적당한 베이스라인)
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.02,
    'num_leaves': 63,
    'max_depth': -1,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_lambda': 1.0,
    'reg_alpha': 0.1,
    'verbose': -1,
}

seeds = [42, 99, 2024]
n_splits = 5

lgbm_oof  = np.zeros(len(X))
lgbm_test = np.zeros(len(X_test))

for seed in seeds:
    print(f"\n{'='*50}")
    print(f"  LightGBM Seed {seed}")
    print(f"{'='*50}")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    seed_aucs = []
    seed_oof = np.zeros(len(X))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        params = lgb_params.copy()
        params['seed'] = seed
        params['feature_fraction_seed'] = seed
        params['bagging_seed'] = seed

        train_set = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        val_set   = lgb.Dataset(X.iloc[val_idx], label=y.iloc[val_idx])

        model = lgb.train(
            params,
            train_set,
            num_boost_round=3000,
            valid_sets=[val_set],
            callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)],
        )

        val_prob = model.predict(X.iloc[val_idx], num_iteration=model.best_iteration)
        seed_oof[val_idx] = val_prob
        lgbm_test += model.predict(X_test, num_iteration=model.best_iteration) / (n_splits * len(seeds))

        auc = roc_auc_score(y.iloc[val_idx], val_prob)
        seed_aucs.append(auc)
        print(f"  Fold {fold+1}: AUC={auc:.5f}, best_iter={model.best_iteration}")

    seed_oof_auc = roc_auc_score(y, seed_oof)
    print(f"▶ Seed {seed} fold-mean: {np.mean(seed_aucs):.5f}, OOF: {seed_oof_auc:.5f}")
    lgbm_oof += seed_oof / len(seeds)

print(f"\n{'='*50}")
print(f"LightGBM 최종 OOF AUC: {roc_auc_score(y, lgbm_oof):.5f}")
print(f"CatBoost 최종 OOF AUC: {roc_auc_score(y, tuned_oof):.5f}")
print(f"{'='*50}")


  LightGBM Seed 42
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[274]	valid_0's auc: 0.738056
  Fold 1: AUC=0.73806, best_iter=274
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[413]	valid_0's auc: 0.742442
  Fold 2: AUC=0.74244, best_iter=413
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[277]	valid_0's auc: 0.740009
  Fold 3: AUC=0.74001, best_iter=277
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[321]	valid_0's auc: 0.738316
  Fold 4: AUC=0.73832, best_iter=321
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[386]	valid_0's auc: 0.740358
  Fold 5: AUC=0.74036, best_iter=386
▶ Seed 42 fold-mean: 0.73984, OOF: 0.73982

  LightGBM Seed 99
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:


In [18]:
from scipy.stats import spearmanr, pearsonr

# 상관 — 0.94~0.96이면 양호 (앙상블 이득 큼)
pear = pearsonr(lgbm_oof, tuned_oof)[0]
spear = spearmanr(lgbm_oof, tuned_oof)[0]
print(f"Pearson  상관 (LGBM vs Cat): {pear:.4f}")
print(f"Spearman 상관 (LGBM vs Cat): {spear:.4f}")
print()

# 단순 평균
simple = (lgbm_oof + tuned_oof) / 2
print(f"단순 평균 (0.5/0.5)  OOF AUC: {roc_auc_score(y, simple):.5f}")

# 그리드 스윕 — Cat 비율
print("\n[가중치 스윕]")
for cat_w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    blend = cat_w * tuned_oof + (1-cat_w) * lgbm_oof
    a = roc_auc_score(y, blend)
    bar = "█" * max(0, int((a - 0.7400) / 0.0001))
    print(f"  Cat={cat_w:.1f}  LGBM={1-cat_w:.1f}  AUC={a:.5f} {bar}")

# 최적 가중치 (Nelder-Mead)
from scipy.optimize import minimize
def neg_auc(w, oofs, y):
    w = np.clip(w, 0, None); s = w.sum()
    if s == 0: return 0.0
    w = w / s
    return -roc_auc_score(y, oofs.T @ w)

oofs = np.vstack([tuned_oof, lgbm_oof])
res = minimize(neg_auc, x0=[0.5, 0.5], args=(oofs, y), method='Nelder-Mead')
w_opt = np.clip(res.x, 0, None); w_opt = w_opt / w_opt.sum()
print(f"\n최적 가중: Cat={w_opt[0]:.3f}, LGBM={w_opt[1]:.3f}, OOF AUC={-res.fun:.5f}")

Pearson  상관 (LGBM vs Cat): 0.9955
Spearman 상관 (LGBM vs Cat): 0.9934

단순 평균 (0.5/0.5)  OOF AUC: 0.74059

[가중치 스윕]
  Cat=0.3  LGBM=0.7  AUC=0.74052 █████
  Cat=0.4  LGBM=0.6  AUC=0.74057 █████
  Cat=0.5  LGBM=0.5  AUC=0.74059 █████
  Cat=0.6  LGBM=0.4  AUC=0.74060 █████
  Cat=0.7  LGBM=0.3  AUC=0.74058 █████

최적 가중: Cat=0.571, LGBM=0.429, OOF AUC=0.74060


In [20]:
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'
os.makedirs(SAVE_DIR, exist_ok=True)

# OOF/test 저장
np.save(f'{SAVE_DIR}/lgbm_oof_v3.npy', lgbm_oof)
np.save(f'{SAVE_DIR}/lgbm_test_v3.npy', lgbm_test)
print(f"✅ lgbm_oof_v3.npy, lgbm_test_v3.npy 저장")

# 최적 가중치로 Cat+LGBM 블렌드 제출 파일
blend_test = w_opt[0] * tuned_test + w_opt[1] * lgbm_test
sample_sub = pd.read_csv('sample_submission.csv')
submission = pd.DataFrame({'ID': test_copy['ID'].values, 'probability': blend_test})
submission = sample_sub[['ID']].merge(submission, on='ID', how='left')
submission.to_csv(f'{SAVE_DIR}/submission_cat_lgbm_v3.csv', index=False)

print(f"✅ submission_cat_lgbm_v3.csv 생성")
print(f"   가중치: Cat={w_opt[0]:.3f}, LGBM={w_opt[1]:.3f}")
print(submission['probability'].describe())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ lgbm_oof_v3.npy, lgbm_test_v3.npy 저장
✅ submission_cat_lgbm_v3.csv 생성
   가중치: Cat=0.571, LGBM=0.429
count    90067.000000
mean         0.258282
std          0.159123
min          0.000720
25%          0.144421
50%          0.269822
75%          0.378572
max          0.759722
Name: probability, dtype: float64


In [23]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

# ============================================================
# 경로 설정
# ============================================================
DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# ============================================================
# OOF 로드
# ============================================================
oof_cat  = tuned_oof                                 # 메모리 (v3 CatBoost)
oof_lgb  = lgbm_oof                                  # 메모리 (LightGBM)
oof_ft   = np.load(f'{DRIVE}/oof_ft.npy')            # 어제 FT OOF
y        = train_copy['임신 성공 여부'].values.astype(int)

# 길이 검증
print(f"y       len={len(y)}")
print(f"oof_cat len={len(oof_cat)}")
print(f"oof_lgb len={len(oof_lgb)}")
print(f"oof_ft  len={len(oof_ft)}")
assert len(oof_cat) == len(oof_lgb) == len(oof_ft) == len(y), "길이 불일치!"

# ============================================================
# 개별 AUC
# ============================================================
auc_cat = roc_auc_score(y, oof_cat)
auc_lgb = roc_auc_score(y, oof_lgb)
auc_ft  = roc_auc_score(y, oof_ft)
print(f"\nCat(v3)        OOF AUC: {auc_cat:.5f}")
print(f"LightGBM       OOF AUC: {auc_lgb:.5f}")
print(f"FT-Transformer OOF AUC: {auc_ft:.5f}")

# ============================================================
# 다양성 (Spearman)
# ============================================================
print("\n[Spearman 상관 — 낮을수록 다양성 ↑]")
print(f"  Cat ↔ LGBM : {spearmanr(oof_cat, oof_lgb).correlation:.4f}")
print(f"  Cat ↔ FT   : {spearmanr(oof_cat, oof_ft).correlation:.4f}  ← 핵심")
print(f"  LGBM ↔ FT  : {spearmanr(oof_lgb, oof_ft).correlation:.4f}  ← 핵심")

# ============================================================
# 최적 가중 (rank 기반)
# ============================================================
ranks = np.vstack([
    rankdata(oof_cat) / len(y),
    rankdata(oof_ft)  / len(y),
    rankdata(oof_lgb) / len(y),
])

# 단순 rank 평균
auc_avg = roc_auc_score(y, ranks.mean(axis=0))
print(f"\n단순 rank 평균 (1/3 each)         OOF AUC: {auc_avg:.5f}")

# Nelder-Mead 최적
def neg_auc(w, oofs, y):
    w = np.clip(w, 0, None); s = w.sum()
    if s == 0: return 0.0
    w = w / s
    return -roc_auc_score(y, oofs.T @ w)

res = minimize(neg_auc, x0=[1/3]*3, args=(ranks, y),
               method='Nelder-Mead',
               options={'xatol':1e-5,'fatol':1e-7,'maxiter':1000})
w = np.clip(res.x, 0, None); w = w / w.sum()
print(f"최적 가중 (rank): Cat={w[0]:.3f}, FT={w[1]:.3f}, LGBM={w[2]:.3f}")
print(f"3-way 최적 OOF AUC: {-res.fun:.5f}")

# 그리드 스윕 (FT 비중 변화)
print("\n[참고] FT 비중 스윕 (Cat:LGBM = 0.6:0.4 고정)")
for ft_w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    rem = 1 - ft_w
    blend = ft_w * ranks[1] + rem * 0.6 * ranks[0] + rem * 0.4 * ranks[2]
    print(f"  FT={ft_w:.1f}  Cat={rem*0.6:.2f}  LGBM={rem*0.4:.2f}  AUC={roc_auc_score(y, blend):.5f}")

# ============================================================
# 결론
# ============================================================
print("\n" + "="*55)
print(f"기준선 (어제 LB 0.74215 → Cat+FT+MLP OOF) : 0.7409")
print(f"이번  Cat(v3)+FT+LGBM 최적 3-way OOF     : {-res.fun:.5f}")
gap = -res.fun - 0.7409
if gap >= 0.0005:
    print(f"✅ +{gap:.5f} 향상! 셀 2로 제출 진행")
elif gap >= -0.0002:
    print(f"≈ {gap:+.5f} (거의 동일). MLP 부활 또는 데이터측 lever 고려")
else:
    print(f"⚠ {gap:+.5f} 하락. 전략 재검토 필요")

# 가중치 저장
np.save(f'{DRIVE}/blend_w_3way_v3.npy', w)
print(f"\n가중치 저장: {DRIVE}/blend_w_3way_v3.npy")

y       len=256351
oof_cat len=256351
oof_lgb len=256351
oof_ft  len=256351

Cat(v3)        OOF AUC: 0.74039
LightGBM       OOF AUC: 0.74025
FT-Transformer OOF AUC: 0.73946

[Spearman 상관 — 낮을수록 다양성 ↑]
  Cat ↔ LGBM : 0.9934
  Cat ↔ FT   : 0.9837  ← 핵심
  LGBM ↔ FT  : 0.9824  ← 핵심

단순 rank 평균 (1/3 each)         OOF AUC: 0.74088
최적 가중 (rank): Cat=0.360, FT=0.303, LGBM=0.337
3-way 최적 OOF AUC: 0.74088

[참고] FT 비중 스윕 (Cat:LGBM = 0.6:0.4 고정)
  FT=0.0  Cat=0.60  LGBM=0.40  AUC=0.74060
  FT=0.1  Cat=0.54  LGBM=0.36  AUC=0.74075
  FT=0.2  Cat=0.48  LGBM=0.32  AUC=0.74085
  FT=0.3  Cat=0.42  LGBM=0.28  AUC=0.74088
  FT=0.4  Cat=0.36  LGBM=0.24  AUC=0.74085
  FT=0.5  Cat=0.30  LGBM=0.20  AUC=0.74077

기준선 (어제 LB 0.74215 → Cat+FT+MLP OOF) : 0.7409
이번  Cat(v3)+FT+LGBM 최적 3-way OOF     : 0.74088
≈ -0.00002 (거의 동일). MLP 부활 또는 데이터측 lever 고려

가중치 저장: /content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤/blend_w_3way_v3.npy


In [27]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# ============================================================
# OOF 4종 로드
# ============================================================
oof_cat = tuned_oof                                  # 메모리 (v3)
oof_lgb = lgbm_oof                                   # 메모리
oof_ft  = np.load(f'{DRIVE}/oof_ft.npy')
oof_mlp = np.load(f'{DRIVE}/oof_mlp.npy')
y       = train_copy['임신 성공 여부'].values.astype(int)

assert len(oof_cat) == len(oof_lgb) == len(oof_ft) == len(oof_mlp) == len(y)

# 개별 AUC
print("="*55)
print("개별 OOF AUC")
print("="*55)
for name, arr in [('Cat(v3)', oof_cat), ('FT', oof_ft),
                  ('LGBM', oof_lgb), ('MLP', oof_mlp)]:
    print(f"  {name:8s}: {roc_auc_score(y, arr):.5f}")

# Spearman 상관 매트릭스
print("\n" + "="*55)
print("Spearman 상관 매트릭스 (낮을수록 다양성↑)")
print("="*55)
oofs_list = [('Cat', oof_cat), ('FT', oof_ft), ('LGBM', oof_lgb), ('MLP', oof_mlp)]
print(f"{'':8s}" + "".join(f"{n:>8s}" for n,_ in oofs_list))
for n1, a1 in oofs_list:
    row = f"{n1:8s}"
    for n2, a2 in oofs_list:
        if n1 == n2:
            row += f"{'  -  ':>8s}"
        else:
            row += f"{spearmanr(a1, a2).correlation:>8.4f}"
    print(row)

# ============================================================
# 4-way 최적 가중 (rank 기반)
# ============================================================
ranks = np.vstack([
    rankdata(oof_cat) / len(y),
    rankdata(oof_ft)  / len(y),
    rankdata(oof_lgb) / len(y),
    rankdata(oof_mlp) / len(y),
])

def neg_auc(w, oofs, y):
    w = np.clip(w, 0, None); s = w.sum()
    if s == 0: return 0.0
    w = w / s
    return -roc_auc_score(y, oofs.T @ w)

# 다중 시작점에서 robust하게 최적화
best_auc, best_w = -1, None
np.random.seed(42)
starts = [np.array([0.25]*4)] + [np.random.dirichlet([1]*4) for _ in range(20)]
for x0 in starts:
    res = minimize(neg_auc, x0=x0, args=(ranks, y),
                   method='Nelder-Mead',
                   options={'xatol':1e-6,'fatol':1e-8,'maxiter':3000})
    if -res.fun > best_auc:
        best_auc, best_w = -res.fun, np.clip(res.x, 0, None) / np.clip(res.x, 0, None).sum()

w4 = best_w
print("\n" + "="*55)
print("4-way 최적 블렌드")
print("="*55)
print(f"가중치  Cat={w4[0]:.3f}  FT={w4[1]:.3f}  LGBM={w4[2]:.3f}  MLP={w4[3]:.3f}")
print(f"4-way OOF AUC: {best_auc:.5f}")
print(f"  vs 3-way(Cat+FT+LGBM)=0.74088 : {best_auc - 0.74088:+.5f}")
print(f"  vs 어제 3-way     =0.7409   : {best_auc - 0.7409:+.5f}")

# 단순 평균도 비교
auc_mean = roc_auc_score(y, ranks.mean(axis=0))
print(f"\n참고) 단순 rank 평균")

개별 OOF AUC
  Cat(v3) : 0.74039
  FT      : 0.73946
  LGBM    : 0.74025
  MLP     : 0.73809

Spearman 상관 매트릭스 (낮을수록 다양성↑)
             Cat      FT    LGBM     MLP
Cat          -    0.9837  0.9934  0.9795
FT        0.9837     -    0.9824  0.9701
LGBM      0.9934  0.9824     -    0.9768
MLP       0.9795  0.9701  0.9768     -  

4-way 최적 블렌드
가중치  Cat=0.248  FT=0.289  LGBM=0.304  MLP=0.159
4-way OOF AUC: 0.74099
  vs 3-way(Cat+FT+LGBM)=0.74088 : +0.00011
  vs 어제 3-way     =0.7409   : +0.00009

참고) 단순 rank 평균


In [31]:
# ============================================================
# 통합 셀: Pseudo-label 생성 (자기완결형 — 외부 파일 의존 없음)
# ============================================================
import numpy as np
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# Drive 로드
oof_ft   = np.load(f'{DRIVE}/oof_ft.npy')
oof_mlp  = np.load(f'{DRIVE}/oof_mlp.npy')
test_ft  = np.load(f'{DRIVE}/test_ft.npy')
test_mlp = np.load(f'{DRIVE}/test_mlp.npy')
y = train_copy['임신 성공 여부'].values.astype(int)

print(f"[Shapes] tuned_test={tuned_test.shape}, lgbm_test={lgbm_test.shape}, "
      f"test_ft={test_ft.shape}, test_mlp={test_mlp.shape}")

# 4-way 가중치 즉석 계산
ranks_oof = np.vstack([
    rankdata(tuned_oof) / len(y),
    rankdata(oof_ft)    / len(y),
    rankdata(lgbm_oof)  / len(y),
    rankdata(oof_mlp)   / len(y),
])

def neg_auc(w, oofs, y):
    w = np.clip(w, 0, None); s = w.sum()
    if s == 0: return 0.0
    w = w / s
    return -roc_auc_score(y, oofs.T @ w)

best_auc, w4 = -1, None
np.random.seed(42)
for x0 in [np.array([0.25]*4)] + [np.random.dirichlet([1]*4) for _ in range(20)]:
    res = minimize(neg_auc, x0=x0, args=(ranks_oof, y),
                   method='Nelder-Mead',
                   options={'xatol':1e-6,'fatol':1e-8,'maxiter':3000})
    if -res.fun > best_auc:
        w = np.clip(res.x, 0, None); w = w / w.sum()
        best_auc, w4 = -res.fun, w

print(f"\n[4-way 가중치 (즉석 계산)]")
print(f"  Cat={w4[0]:.3f}  FT={w4[1]:.3f}  LGBM={w4[2]:.3f}  MLP={w4[3]:.3f}")
print(f"  4-way OOF AUC: {best_auc:.5f}")

np.save(f'{DRIVE}/blend_w_4way_v3.npy', w4)
print(f"  저장: blend_w_4way_v3.npy")

# Teacher 예측 (4-way test 블렌드)
ranks_test = np.vstack([
    rankdata(tuned_test) / len(tuned_test),
    rankdata(test_ft)    / len(test_ft),
    rankdata(lgbm_test)  / len(lgbm_test),
    rankdata(test_mlp)   / len(test_mlp),
])
teacher_pred = ranks_test.T @ w4

# Confident 샘플 선택
HI_PCT, LO_PCT = 0.05, 0.10
hi_thresh = np.quantile(teacher_pred, 1 - HI_PCT)
lo_thresh = np.quantile(teacher_pred, LO_PCT)
pos_idx = np.where(teacher_pred > hi_thresh)[0]
neg_idx = np.where(teacher_pred < lo_thresh)[0]

print(f"\n[Pseudo-label]")
print(f"  Teacher range:  [{teacher_pred.min():.4f}, {teacher_pred.max():.4f}]")
print(f"  Top {HI_PCT*100:>2.0f}% (>{hi_thresh:.4f}) → 1: {len(pos_idx):>6,}개")
print(f"  Bot {LO_PCT*100:>2.0f}% (<{lo_thresh:.4f}) → 0: {len(neg_idx):>6,}개")
print(f"  Total pseudo:   {len(pos_idx)+len(neg_idx):>6,} ({(len(pos_idx)+len(neg_idx))/len(train_copy)*100:.2f}% 증대)")
print(f"  Pseudo pos rate: {len(pos_idx)/(len(pos_idx)+len(neg_idx)):.3f}")

[Shapes] tuned_test=(90067,), lgbm_test=(90067,), test_ft=(90067,), test_mlp=(90067,)

[4-way 가중치 (즉석 계산)]
  Cat=0.248  FT=0.289  LGBM=0.304  MLP=0.159
  4-way OOF AUC: 0.74099
  저장: blend_w_4way_v3.npy

[Pseudo-label]
  Teacher range:  [0.0032, 1.0000]
  Top  5% (>0.9477) → 1:  4,504개
  Bot 10% (<0.0898) → 0:  9,007개
  Total pseudo:   13,511 (5.27% 증대)
  Pseudo pos rate: 0.333


In [32]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ============================================================
# 데이터 준비 (v3 학습 셀과 동일한 X, y 구조)
# ============================================================
X      = train_copy.drop(columns=['ID', '임신 성공 여부'])
y      = train_copy['임신 성공 여부']
X_test = test_copy.drop(columns=['ID'])

# Pseudo 데이터 (DataFrame으로 — 원본과 같은 형식)
pseudo_idx = np.concatenate([pos_idx, neg_idx])
X_pseudo   = test_copy.iloc[pseudo_idx].drop(columns=['ID']).reset_index(drop=True)
y_pseudo   = pd.Series(
    np.concatenate([np.ones(len(pos_idx)), np.zeros(len(neg_idx))]).astype(int)
)

# 컬럼 일치 검증
assert list(X.columns) == list(X_pseudo.columns) == list(X_test.columns), \
    f"컬럼 불일치!\nX: {len(X.columns)}, X_pseudo: {len(X_pseudo.columns)}, X_test: {len(X_test.columns)}"

print(f"X shape:        {X.shape}")
print(f"X_pseudo shape: {X_pseudo.shape}")
print(f"X_test shape:   {X_test.shape}")
print(f"Pseudo: pos={int(y_pseudo.sum())}, neg={int((1-y_pseudo).sum())}")

# ============================================================
# 3-seed × 5-fold (v3 학습 셀과 100% 동일 구조, cat_features 안 씀)
# ============================================================
seeds = [42, 99, 2024]
n_splits = 5

oof_pseudo  = np.zeros(len(X))
test_pseudo = np.zeros(len(X_test))

for seed in seeds:
    print(f"\n=== Seed {seed} ===")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    seed_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        params = best_params.copy()
        params['random_seed'] = seed

        # 핵심: 학습 = 원본 train_fold + 모든 pseudo (DataFrame concat)
        X_tr = pd.concat([X.iloc[tr_idx], X_pseudo], axis=0).reset_index(drop=True)
        y_tr = pd.concat([y.iloc[tr_idx], y_pseudo], axis=0).reset_index(drop=True)
        # 검증 = 원본 valid_fold만
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        m = CatBoostClassifier(**params)   # cat_features 안 넘김 (원본 그대로)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va))

        val_prob = m.predict_proba(X_va)[:, 1]
        oof_pseudo[val_idx] += val_prob / len(seeds)
        test_pseudo += m.predict_proba(X_test)[:, 1] / (n_splits * len(seeds))

        auc = roc_auc_score(y_va, val_prob)
        seed_aucs.append(auc)
        print(f"  fold {fold}: AUC = {auc:.5f}")

    print(f"  Seed {seed} 평균: {np.mean(seed_aucs):.5f}")

# ============================================================
# 결과
# ============================================================
final_auc = roc_auc_score(y, oof_pseudo)
print("\n" + "="*55)
print(f"▶ Pseudo-Cat(v3) OOF AUC: {final_auc:.5f}")
print(f"▶ 기준선 (Cat-v3 단독)   : 0.74039")
print(f"▶ 향상: {final_auc - 0.74039:+.5f}")
print("="*55)

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'
np.save(f'{DRIVE}/oof_cat_pseudo.npy',  oof_pseudo)
np.save(f'{DRIVE}/test_cat_pseudo.npy', test_pseudo)
print(f"\n저장: oof_cat_pseudo.npy, test_cat_pseudo.npy")

X shape:        (256351, 128)
X_pseudo shape: (13511, 128)
X_test shape:   (90067, 128)
Pseudo: pos=4504, neg=9007

=== Seed 42 ===
  fold 0: AUC = 0.73844
  fold 1: AUC = 0.74348
  fold 2: AUC = 0.74093
  fold 3: AUC = 0.73845
  fold 4: AUC = 0.74131
  Seed 42 평균: 0.74052

=== Seed 99 ===
  fold 0: AUC = 0.74053
  fold 1: AUC = 0.74055
  fold 2: AUC = 0.73808
  fold 3: AUC = 0.73940
  fold 4: AUC = 0.74309
  Seed 99 평균: 0.74033

=== Seed 2024 ===
  fold 0: AUC = 0.74272
  fold 1: AUC = 0.74048
  fold 2: AUC = 0.74149
  fold 3: AUC = 0.73913
  fold 4: AUC = 0.73895
  Seed 2024 평균: 0.74056

▶ Pseudo-Cat(v3) OOF AUC: 0.74067
▶ 기준선 (Cat-v3 단독)   : 0.74039
▶ 향상: +0.00028

저장: oof_cat_pseudo.npy, test_cat_pseudo.npy


In [42]:
# === 복구 + 최종 블렌드 비교 + 제출 (1분 이내) ===
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'
TRAIN_CSV = '/content/train.csv'
SUB_CSV   = '/content/sample_submission.csv'

# === y 복구 ===
train = pd.read_csv(TRAIN_CSV)
train = train.dropna(subset=['임신 성공 여부']).reset_index(drop=True)
y = train['임신 성공 여부'].values.astype(int)
print(f"y shape: {y.shape}, positive rate: {y.mean():.4f}")

# === 모든 OOF/test 로드 ===
tuned_oof   = np.load(f'{DRIVE}/tuned_oof.npy')
tuned_test  = np.load(f'{DRIVE}/tuned_test.npy')
oof_pseudo  = np.load(f'{DRIVE}/oof_cat_pseudo.npy')
test_pseudo = np.load(f'{DRIVE}/test_cat_pseudo.npy')
oof_ft      = np.load(f'{DRIVE}/oof_ft.npy')
test_ft     = np.load(f'{DRIVE}/test_ft.npy')
oof_mlp     = np.load(f'{DRIVE}/oof_mlp.npy')
test_mlp    = np.load(f'{DRIVE}/test_mlp.npy')
lgbm_oof    = np.load(f'{DRIVE}/lgbm_oof_v3.npy')
lgbm_test   = np.load(f'{DRIVE}/lgbm_test_v3.npy')

# 길이 확인
for name, arr in [('tuned_oof', tuned_oof), ('oof_pseudo', oof_pseudo),
                  ('oof_ft', oof_ft), ('oof_mlp', oof_mlp), ('lgbm_oof', lgbm_oof)]:
    assert len(arr) == len(y), f"{name} 길이 불일치: {len(arr)} vs {len(y)}"

print("\n[개별 OOF AUC]")
for name, arr in [('Cat-v3', tuned_oof), ('Cat-pseudo', oof_pseudo),
                  ('FT', oof_ft), ('LGBM', lgbm_oof), ('MLP', oof_mlp)]:
    print(f"  {name:11s}: {roc_auc_score(y, arr):.5f}")

# === 빠른 블렌드 최적화 ===
def opt_blend(oofs_list, y, n_starts=10):
    ranks = np.vstack([rankdata(o)/len(y) for o in oofs_list])
    def neg_auc(w):
        w = np.clip(w, 0, None); s = w.sum()
        if s == 0: return 0.0
        return -roc_auc_score(y, ranks.T @ (w/s))
    best_auc, best_w = -1, None
    np.random.seed(42)
    starts = [np.array([1/len(oofs_list)]*len(oofs_list))] + \
             [np.random.dirichlet([1]*len(oofs_list)) for _ in range(n_starts)]
    for x0 in starts:
        res = minimize(neg_auc, x0, method='Nelder-Mead',
                       options={'xatol':1e-5,'fatol':1e-7,'maxiter':1000})
        if -res.fun > best_auc:
            w = np.clip(res.x, 0, None); w = w/w.sum()
            best_auc, best_w = -res.fun, w
    return best_auc, best_w

# === 5 변형 비교 ===
print("\n" + "="*70)
print("블렌드 변형 비교")
print("="*70)

configs = [
    ('A: Cat-v3 + FT + MLP (no pseudo)',
     [tuned_oof, oof_ft, oof_mlp],
     [tuned_test, test_ft, test_mlp],
     ['Cat', 'FT', 'MLP']),
    ('B: Cat-pseudo + FT + MLP',
     [oof_pseudo, oof_ft, oof_mlp],
     [test_pseudo, test_ft, test_mlp],
     ["Cat'", 'FT', 'MLP']),
    ('C: Cat-pseudo + FT + LGBM + MLP',
     [oof_pseudo, oof_ft, lgbm_oof, oof_mlp],
     [test_pseudo, test_ft, lgbm_test, test_mlp],
     ["Cat'", 'FT', 'LGBM', 'MLP']),
    ('D: Cat + Cat-pseudo + FT + MLP',
     [tuned_oof, oof_pseudo, oof_ft, oof_mlp],
     [tuned_test, test_pseudo, test_ft, test_mlp],
     ['Cat', "Cat'", 'FT', 'MLP']),
    ('E: Cat + Cat-pseudo + FT + LGBM + MLP',
     [tuned_oof, oof_pseudo, oof_ft, lgbm_oof, oof_mlp],
     [tuned_test, test_pseudo, test_ft, lgbm_test, test_mlp],
     ['Cat', "Cat'", 'FT', 'LGBM', 'MLP']),
]

variants = []
for name, oofs, tests, lbls in configs:
    auc, w = opt_blend(oofs, y, n_starts=10)
    variants.append((name, auc, w, oofs, tests, lbls))
    weights_str = "  ".join(f"{l}={v:.3f}" for l, v in zip(lbls, w))
    print(f"\n{name}")
    print(f"  OOF: {auc:.5f}   ({weights_str})")

# === 최고 변형 선택 + 제출 ===
best_idx = max(range(len(variants)), key=lambda i: variants[i][1])
best_name, best_auc, best_w, _, best_tests, best_lbls = variants[best_idx]

print("\n" + "="*70)
print(f" 최고: {best_name}")
print(f"   OOF: {best_auc:.5f}")
print(f"   가중치: {dict(zip(best_lbls, best_w.round(3).tolist()))}")
print(f"   기존 4-way (no pseudo): 0.74099  →  {best_auc - 0.74099:+.5f}")
print("="*70)

# 최고 변형 제출 파일
test_ranks = np.vstack([rankdata(t)/len(t) for t in best_tests])
final_pred = test_ranks.T @ best_w

sub = pd.read_csv(SUB_CSV)
sub['probability'] = final_pred
best_path = f'{DRIVE}/submission_pseudo_best_{chr(65+best_idx)}.csv'
sub.to_csv(best_path, index=False)
print(f"\n저장: submission_pseudo_best_{chr(65+best_idx)}.csv")
print(sub['probability'].describe())

# B 변형 백업 (검증된 3-way 레시피 + pseudo)
if best_idx != 1:
    _, _, w_b, _, b_tests, b_lbls = variants[1]
    test_ranks_b = np.vstack([rankdata(t)/len(t) for t in b_tests])
    sub2 = pd.read_csv(SUB_CSV)
    sub2['probability'] = test_ranks_b.T @ w_b
    sub2.to_csv(f'{DRIVE}/submission_pseudo_B_3way.csv', index=False)
    print(f"백업: submission_pseudo_B_3way.csv (B 변형)")

y shape: (256351,), positive rate: 0.2583

[개별 OOF AUC]
  Cat-v3     : 0.74044
  Cat-pseudo : 0.74067
  FT         : 0.73946
  LGBM       : 0.74025
  MLP        : 0.73809

블렌드 변형 비교

A: Cat-v3 + FT + MLP (no pseudo)
  OOF: 0.74091   (Cat=0.514  FT=0.314  MLP=0.172)

B: Cat-pseudo + FT + MLP
  OOF: 0.74105   (Cat'=0.568  FT=0.286  MLP=0.146)

C: Cat-pseudo + FT + LGBM + MLP
  OOF: 0.74109   (Cat'=0.406  FT=0.254  LGBM=0.210  MLP=0.130)

D: Cat + Cat-pseudo + FT + MLP
  OOF: 0.74105   (Cat=0.000  Cat'=0.566  FT=0.286  MLP=0.147)

E: Cat + Cat-pseudo + FT + LGBM + MLP
  OOF: 0.74109   (Cat=0.000  Cat'=0.411  FT=0.251  LGBM=0.205  MLP=0.133)

 최고: C: Cat-pseudo + FT + LGBM + MLP
   OOF: 0.74109
   가중치: {"Cat'": 0.406, 'FT': 0.254, 'LGBM': 0.21, 'MLP': 0.13}
   기존 4-way (no pseudo): 0.74099  →  +0.00010

저장: submission_pseudo_best_C.csv
count    90067.000000
mean         0.500006
std          0.287597
min          0.002911
25%          0.252410
50%          0.500749
75%          0.748165
ma

In [45]:
# ============================================================
# 어드버서리얼 검증
# ============================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# === 데이터 로드 (v2 parquet — 분포 진단용으로 충분) ===
train_proc = pd.read_parquet(f'{DRIVE}/train_processed.parquet')
test_proc  = pd.read_parquet(f'{DRIVE}/test_processed.parquet')

print(f"train shape: {train_proc.shape}")
print(f"test  shape: {test_proc.shape}")

# Feature 컬럼 정의
exclude = ['ID', '임신 성공 여부']
feature_cols = [c for c in train_proc.columns if c not in exclude and c in test_proc.columns]
print(f"feature 수: {len(feature_cols)}")

# === Train(0) vs Test(1) 라벨 만들기 ===
X_combined = pd.concat([train_proc[feature_cols], test_proc[feature_cols]], axis=0).reset_index(drop=True)
y_combined = np.concatenate([np.zeros(len(train_proc)), np.ones(len(test_proc))]).astype(int)

print(f"\nX_combined shape: {X_combined.shape}")
print(f"y_combined: train={int((y_combined==0).sum()):,}, test={int((y_combined==1).sum()):,}")

# === 5-fold 어드버서리얼 분류기 ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_adv = np.zeros(len(X_combined))
feature_importances = np.zeros(len(feature_cols))

print("\n[학습 진행]")
for fold, (tr, va) in enumerate(skf.split(X_combined, y_combined)):
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        random_state=42,
        verbose=-1,
    )
    model.fit(
        X_combined.iloc[tr], y_combined[tr],
        eval_set=[(X_combined.iloc[va], y_combined[va])],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    oof_adv[va] = model.predict_proba(X_combined.iloc[va])[:, 1]
    feature_importances += model.feature_importances_ / 5
    fold_auc = roc_auc_score(y_combined[va], oof_adv[va])
    print(f"  fold {fold}: AUC = {fold_auc:.5f}")

adv_auc = roc_auc_score(y_combined, oof_adv)

# === 결과 해석 ===
print("\n" + "="*60)
print(f"어드버서리얼 AUC: {adv_auc:.5f}")
print("="*60)
if adv_auc < 0.52:
    print("✅ 매우 비슷함 — 분포 일치, OOF-LB gap은 다른 원인")
    print("   다음 lever: stacking, calibration, multi-iteration pseudo")
elif adv_auc < 0.58:
    print("✅ 분포 비슷함 — 약한 shift, OOF-LB gap의 주요 원인 아님")
    print("   상위 feature 1~2개 제거 정도 시도 가능")
elif adv_auc < 0.68:
    print("⚠  중간 정도 shift — OOF-LB gap의 한 원인일 수 있음")
    print("   → 상위 feature 분석 후 제거/가중치 학습 시도")
else:
    print("❌ 큰 shift — 진짜 문제!")
    print("   → adversarial weighting 또는 covariate shift correction 필요")

# === Top feature (분포 차이 큰 것) ===
print("\n[Train/Test 구분에 기여하는 feature TOP 20]")
print("  (importance 높을수록 train/test 분포 다름)")
imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importances,
    'train_mean': [train_proc[c].mean() for c in feature_cols],
    'test_mean':  [test_proc[c].mean() for c in feature_cols],
})
imp_df['mean_diff_pct'] = ((imp_df['test_mean'] - imp_df['train_mean']) /
                            (imp_df['train_mean'].abs() + 1e-9) * 100).round(2)
imp_df = imp_df.sort_values('importance', ascending=False).head(20)

print(f"  {'importance':>10s}  {'mean_diff%':>10s}  feature")
print(f"  {'-'*10}  {'-'*10}  {'-'*40}")
for _, row in imp_df.iterrows():
    print(f"  {row['importance']:>10.1f}  {row['mean_diff_pct']:>+10.1f}  {row['feature']}")

# 결과 저장
imp_df.to_csv(f'{DRIVE}/adversarial_importance.csv', index=False)
print(f"\n저장: adversarial_importance.csv")

train shape: (256351, 104)
test  shape: (90067, 103)
feature 수: 102

X_combined shape: (346418, 102)
y_combined: train=256,351, test=90,067

[학습 진행]
  fold 0: AUC = 0.49849
  fold 1: AUC = 0.50391
  fold 2: AUC = 0.50495
  fold 3: AUC = 0.50104
  fold 4: AUC = 0.50519

어드버서리얼 AUC: 0.50292
✅ 매우 비슷함 — 분포 일치, OOF-LB gap은 다른 원인
   다음 lever: stacking, calibration, multi-iteration pseudo

[Train/Test 구분에 기여하는 feature TOP 20]
  (importance 높을수록 train/test 분포 다름)
  importance  mean_diff%  feature
  ----------  ----------  ----------------------------------------
        25.2        -0.5  신선난자_배아효율
        22.0        -0.2  난자_혼합률
        20.0        +0.4  혼합난자_배아효율
        19.8        +0.3  배아_여유
        17.4        +0.3  시술 시기 코드
        12.4        -0.0  클리닉 내 총 시술 횟수
        11.0        -0.7  저장된 배아 수
        10.4        -0.2  시술 당시 나이
        10.0        -0.7  파이프_효율
         9.8        +0.1  미세주입에서 생성된 배아 수
         9.4        +0.2  파트너 정자와 혼합된 난자 수
         9.4        -0.5  이식_비율
       

In [46]:
# ==========================================================
# Stacking with LR meta-learner (Cat-pseudo + FT + MLP + LGBM)
# ==========================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from scipy.optimize import minimize

DRIVE = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# ---------- 1. Load OOF / Test ----------
oof_cat   = np.load(f'{DRIVE}/oof_cat_pseudo.npy')
test_cat  = np.load(f'{DRIVE}/test_cat_pseudo.npy')
oof_ft    = np.load(f'{DRIVE}/oof_ft.npy')
test_ft   = np.load(f'{DRIVE}/test_ft.npy')
oof_mlp   = np.load(f'{DRIVE}/oof_mlp.npy')
test_mlp  = np.load(f'{DRIVE}/test_mlp.npy')
oof_lgbm  = np.load(f'{DRIVE}/lgbm_oof_v3.npy')
test_lgbm = np.load(f'{DRIVE}/lgbm_test_v3.npy')

# ---------- 2. Label ----------
train = pd.read_csv('/content/train.csv')
label_candidates = ['임신 성공 여부', '임신성공여부', 'target', 'label', 'y']
label_col = next(c for c in label_candidates if c in train.columns)
y = train[label_col].values
print(f"Label='{label_col}', pos_rate={y.mean():.4f}, n={len(y)}")

print("\n[Individual OOF AUCs]")
for name, arr in [('Cat-pseudo', oof_cat), ('FT', oof_ft),
                  ('MLP', oof_mlp), ('LGBM', oof_lgbm)]:
    print(f"  {name:12s}: {roc_auc_score(y, arr):.5f}")

# ---------- 3. Build matrices + transforms ----------
oof_mat  = np.column_stack([oof_cat, oof_ft, oof_mlp, oof_lgbm])
test_mat = np.column_stack([test_cat, test_ft, test_mlp, test_lgbm])
names = ['Cat-pseudo', 'FT', 'MLP', 'LGBM']

def to_logit(p, eps=1e-7):
    p = np.clip(p, eps, 1-eps)
    return np.log(p / (1-p))

def rank_norm(arr):
    out = np.zeros_like(arr)
    for j in range(arr.shape[1]):
        out[:, j] = rankdata(arr[:, j]) / (arr.shape[0] + 1)
    return out

oof_logit, test_logit = to_logit(oof_mat), to_logit(test_mat)
oof_rank,  test_rank  = rank_norm(oof_mat), rank_norm(test_mat)

# ---------- 4. Baseline: weighted blend (current C-variant 방식) ----------
def neg_auc_w(w, M, y):
    w = np.abs(w); s = w.sum()
    if s == 0: return 0
    return -roc_auc_score(y, M @ (w/s))

best_w, best_w_auc = None, -np.inf
for seed in range(10):
    np.random.seed(seed)
    w0 = np.random.dirichlet(np.ones(4))
    res = minimize(neg_auc_w, w0, args=(oof_mat, y),
                   method='Nelder-Mead', options={'maxiter': 1000, 'xatol': 1e-7})
    if -res.fun > best_w_auc:
        best_w_auc, best_w = -res.fun, np.abs(res.x)/np.abs(res.x).sum()

blend_test = test_mat @ best_w
print(f"\n[Weighted blend baseline]  OOF={best_w_auc:.5f}")
print(f"  weights: {dict(zip(names, np.round(best_w,4)))}")

# ---------- 5. Stacking: LR meta-learner, 5-fold honest OOF ----------
def stack_lr(F_oof, F_test, y, C=1.0, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_s  = np.zeros(len(y))
    test_s = np.zeros(len(F_test))
    for tr, va in skf.split(F_oof, y):
        lr = LogisticRegression(C=C, solver='lbfgs', max_iter=1000)
        lr.fit(F_oof[tr], y[tr])
        oof_s[va] = lr.predict_proba(F_oof[va])[:, 1]
        test_s   += lr.predict_proba(F_test)[:, 1] / n_splits
    return oof_s, test_s

configs = []
for repr_name, F_oof, F_test in [('raw', oof_mat, test_mat),
                                  ('logit', oof_logit, test_logit),
                                  ('rank',  oof_rank,  test_rank)]:
    for C in [0.01, 0.1, 1.0, 10.0]:
        oof_s, test_s = stack_lr(F_oof, F_test, y, C=C)
        configs.append((repr_name, C, roc_auc_score(y, oof_s), oof_s, test_s))

print(f"\n[Stacking LR — all configs]")
print(f"  {'repr':<6}{'C':<8}{'OOF AUC':<10}{'Δ vs blend':<10}")
for r, C, auc, *_ in configs:
    print(f"  {r:<6}{C:<8.2f}{auc:.5f}   {auc - best_w_auc:+.5f}")

# ---------- 6. Best config ----------
best = max(configs, key=lambda x: x[2])
print(f"\n[Best stacking]  repr={best[0]}, C={best[1]}, OOF={best[2]:.5f}")
print(f"  Δ vs weighted blend: {best[2] - best_w_auc:+.5f}")

# Full-data fit으로 LR 계수 해석
repr_map = {'raw': (oof_mat, test_mat),
            'logit': (oof_logit, test_logit),
            'rank':  (oof_rank,  test_rank)}
F_oof_b, F_test_b = repr_map[best[0]]
lr_full = LogisticRegression(C=best[1], solver='lbfgs', max_iter=1000)
lr_full.fit(F_oof_b, y)
print(f"\n[LR coefficients - {best[0]} repr]")
for n, c in zip(names, lr_full.coef_[0]):
    print(f"  {n:12s}: {c:+.4f}")
print(f"  intercept   : {lr_full.intercept_[0]:+.4f}")

abs_c = np.abs(lr_full.coef_[0])
print(f"\n[Stacking 암묵적 가중치 (|coef|/sum)]")
for n, w in zip(names, abs_c/abs_c.sum()):
    print(f"  {n:12s}: {w:.4f}")

# ---------- 7. 진단 해석 ----------
delta = best[2] - best_w_auc
print(f"\n[진단]")
if delta > -0.0002:
    print(f"  Δ={delta:+.5f} → stacking이 weighted blend와 거의 동등하거나 더 좋음.")
    print(f"  → 가중치가 OOF에 과적합되지 않았을 가능성 ↑.")
    print(f"  → LB에서도 LR stacking이 비슷하거나 약간 나을 가능성.")
else:
    print(f"  Δ={delta:+.5f} → stacking이 weighted blend보다 낮음.")
    print(f"  → Nelder-Mead가 OOF에 약간 과적합한 신호일 수 있음 (LR이 정규화로 못 따라감).")
    print(f"  → LB에서 stacking이 오히려 더 잘 나올 가능성 있음 (덜 과적합).")

# ---------- 8. Save submission ----------
sub = pd.read_csv('/content/sample_submission.csv')
target_col = sub.columns[1]
sub[target_col] = best[4]
out = f'{DRIVE}/submission_stacking_lr.csv'
sub.to_csv(out, index=False)
np.save(f'{DRIVE}/oof_stacking_lr.npy',  best[3])
np.save(f'{DRIVE}/test_stacking_lr.npy', best[4])
print(f"\n[Saved] {out}")
print(f"  pred range: [{best[4].min():.4f}, {best[4].max():.4f}], mean={best[4].mean():.4f}")

Label='임신 성공 여부', pos_rate=0.2583, n=256351

[Individual OOF AUCs]
  Cat-pseudo  : 0.74067
  FT          : 0.73946
  MLP         : 0.73809
  LGBM        : 0.74025

[Weighted blend baseline]  OOF=0.74113
  weights: {'Cat-pseudo': np.float64(0.4019), 'FT': np.float64(0.2542), 'MLP': np.float64(0.1378), 'LGBM': np.float64(0.2062)}

[Stacking LR — all configs]
  repr  C       OOF AUC   Δ vs blend
  raw   0.01    0.74097   -0.00016
  raw   0.10    0.74037   -0.00076
  raw   1.00    0.73978   -0.00135
  raw   10.00   0.73964   -0.00149
  logit 0.01    0.74105   -0.00008
  logit 0.10    0.74103   -0.00009
  logit 1.00    0.74104   -0.00009
  logit 10.00   0.74104   -0.00009
  rank  0.01    0.74101   -0.00012
  rank  0.10    0.74100   -0.00013
  rank  1.00    0.74100   -0.00013
  rank  10.00   0.74100   -0.00013

[Best stacking]  repr=logit, C=0.01, OOF=0.74105
  Δ vs weighted blend: -0.00008

[LR coefficients - logit repr]
  Cat-pseudo  : +0.2421
  FT          : +0.2900
  MLP         : +0.174